In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from getpass import getpass
GIT_TOKEN = getpass('Enter your GitHub Personal Access Token (ghp_...): ')
GIT_USERNAME = "navdeepsingh1609"  # <-- your GitHub username
REPO_NAME = "CSC2529-Fall-Project"
REPO_URL = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{REPO_NAME}.git"

Enter your GitHub Personal Access Token (ghp_...): ··········


In [ ]:
# Clone repo
%cd /content
!git clone {REPO_URL}
%cd {REPO_NAME}
!git checkout model2
# Init MambaIR submodule
!git submodule update --init --force --recursive --remote

# Install deps
!pip install -q -r requirements.txt

/content
Cloning into 'CSC2529-Fall-Project'...
remote: Enumerating objects: 415, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 415 (delta 8), reused 19 (delta 4), pack-reused 382 (from 1)
Receiving objects: 100% (415/415), 183.14 MiB | 15.90 MiB/s, done.
Resolving deltas: 100% (158/158), done.
Updating files: 100% (35/35), done.
/content/CSC2529-Fall-Project
Branch 'model2' set up to track remote branch 'model2' from 'origin'.
Switched to a new branch 'model2'
Submodule 'models/external/MambaIR' (https://github.com/csguoh/MambaIR.git) registered for path 'models/external/MambaIR'
Cloning into '/content/CSC2529-Fall-Project/models/external/MambaIR'...
Submodule path 'models/external/MambaIR': checked out '863ca48768a0a6da636bb354f585ef7cfa3fd558'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172

In [ ]:
!mkdir -p /content/dataset
!ln -s "/content/drive/MyDrive/Computational Imaging Project/UDC-SIT/UDC-SIT" "/content/dataset/UDC-SIT"

In [ ]:
# Create subset
!python scripts/create_subset.py
!ls data/UDC-SIT_subset/train/input | head
!ls data/UDC-SIT_subset/val/input | head

Creating training subset...
Copying 30 files from /content/drive/MyDrive/Computational Imaging Project/UDC-SIT/UDC-SIT/training/input...
Creating validation subset...
Copying 7 files from /content/drive/MyDrive/Computational Imaging Project/UDC-SIT/UDC-SIT/validation/input...
Subset creation complete.
1000.npy
1001.npy
1002.npy
1003.npy
1008.npy
1009.npy
100.npy
1010.npy
1011.npy
1012.npy
1004.npy
1027.npy
1034.npy
1081.npy
1127.npy
1132.npy
114.npy


In [ ]:
# Quick teacher training on subset
!python train_teacher_quick.py

--- [train_teacher_quick] Setting up system path...
--- [train_teacher_quick] Added /content/CSC2529-Fall-Project/models/external/MambaIR to sys.path
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

--- [train_teacher_quick] Configuration ---
Train Dir: data/UDC-SIT_subset/train
Val Dir:   data/UDC-SIT_subset/val
Patch Size: 256, Batch Size: 2
Epochs: 10, Learning Rate: 0.0001
Device: cuda
Checkpoint: teacher_quick_1epoch.pth
-----------------------------------

--- [train_teacher_quick] Loading datasets...
--- [UDCDataset] Loaded 30 files from data/UDC-SIT_subset/train
--- [UDCDataset] Loaded 7 files from data/UDC-SIT_subset/val
--- [train_teacher_quick] Initializing model...
--- [Teacher] Initializing with 4 in-channels and 4 out-channels.
--- [FrequencyDomainB

In [ ]:
# Quick KD student training on subset
!python train_student_kd_quick.py

--- [train_student_kd_quick] Setting up system path...
--- [train_student_kd_quick] Added /content/CSC2529-Fall-Project/models/external/MambaIR to sys.path
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

--- [train_student_kd_quick] Configuration ---
Train Dir: data/UDC-SIT_subset/train
Val Dir:   data/UDC-SIT_subset/val
Patch Size: 256, Batch Size: 16
Epochs: 12, Learning Rate: 0.0002
Device: cuda
Teacher Weights: teacher_quick_1epoch.pth
Student Save Path: student_quick_1epoch.pth
-----------------------------------

--- [train_student_kd_quick] Loading datasets...
--- [UDCDataset] Loaded 30 files from data/UDC-SIT_subset/train
--- [UDCDataset] Loaded 7 files from data/UDC-SIT_subset/val
--- [train_student_kd_quick] Loading teacher from teacher_quick_1epoch.pt

In [ ]:
# Quick patch-based evaluation
!python testing_quick.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
--- [testing_quick] Split: val, Num images: 8
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.w

In [ ]:
# Copy dataset tarballs to local SSD
!mkdir -p /content/dataset
!cp "/content/drive/MyDrive/Computational Imaging Project/UDC-SIT/UDC-SIT"-*.tar.gz /content/dataset/

# Extract training/validation
%cd /content/dataset
!cat UDC-SIT-*.tar.gz | tar -xzvf -
%cd /content/CSC2529-Fall-Project

/content/dataset
UDC-SIT/
UDC-SIT/validation/
UDC-SIT/validation/GT/
UDC-SIT/validation/GT/1257.npy
UDC-SIT/validation/GT/2491.npy
UDC-SIT/validation/GT/1398.npy
UDC-SIT/validation/GT/2889.npy
UDC-SIT/validation/GT/2585.npy
UDC-SIT/validation/GT/2561.npy
UDC-SIT/validation/GT/597.npy
UDC-SIT/validation/GT/2777.npy
UDC-SIT/validation/GT/210.npy
UDC-SIT/validation/GT/2581.npy
UDC-SIT/validation/GT/2783.npy
UDC-SIT/validation/GT/2762.npy
UDC-SIT/validation/GT/1330.npy
UDC-SIT/validation/GT/2716.npy
UDC-SIT/validation/GT/27.npy
UDC-SIT/validation/GT/2541.npy
UDC-SIT/validation/GT/418.npy
UDC-SIT/validation/GT/2537.npy
UDC-SIT/validation/GT/2582.npy
UDC-SIT/validation/GT/2276.npy
UDC-SIT/validation/GT/2639.npy
UDC-SIT/validation/GT/1613.npy
UDC-SIT/validation/GT/2587.npy
UDC-SIT/validation/GT/2590.npy
UDC-SIT/validation/GT/1334.npy
UDC-SIT/validation/GT/2511.npy
UDC-SIT/validation/GT/91.npy
UDC-SIT/validation/GT/2547.npy
UDC-SIT/validation/GT/2562.npy
UDC-SIT/validation/GT/2704.npy
UDC-SIT/

In [ ]:
%run train_teacher.py

--- [train_teacher] Setting up system path...
--- [train_teacher] Added /content/CSC2529-Fall-Project/models/external/MambaIR to sys.path


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)



--- [train_teacher] Configuration ---
Train Dir: /content/dataset/UDC-SIT/training
Val Dir:   /content/dataset/UDC-SIT/validation
Patch Size: 256, Batch Size: 8
Epochs: 22, Learning Rate: 0.0001
Device: cuda
Local latest ckpt: teacher_4ch_latest.pth
Drive latest ckpt: /content/drive/MyDrive/Computational Imaging Project/checkpoints/teacher_latest.pth
-----------------------------------

--- [train_teacher] Loading datasets...
--- [UDCDataset] Loaded 1864 files from /content/dataset/UDC-SIT/training
--- [UDCDataset] Loaded 238 files from /content/dataset/UDC-SIT/validation
--- [train_teacher] Device: cuda
--- [train_teacher] Train samples: 1864, Val samples: 238
--- [train_teacher] Initializing model...
--- [Teacher] Initializing with 4 in-channels and 4 out-channels.
--- [FrequencyDomainBlock] Initialized with in_channels=4.
--- [Teacher] Model initialized (MambaIR + Freq residual gating).
--- [train_teacher] Initializing losses...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


...

--- [train_teacher] Epoch 21 Val Charbonnier Loss: 0.0400
Saved latest teacher checkpoint to teacher_4ch_latest.pth and /content/drive/MyDrive/Computational Imaging Project/checkpoints/teacher_latest.pth


Epoch 22/22: 100%|██████████| 233/233 [10:50<00:00,  2.79s/it]

--- [train_teacher] Epoch 22 Train Loss: 0.1218


--- [train_teacher] Epoch 22 Val Charbonnier Loss: 0.0402
Saved latest teacher checkpoint to teacher_4ch_latest.pth and /content/drive/MyDrive/Computational Imaging Project/checkpoints/teacher_latest.pth
--- [train_teacher] Final model saved as teacher_4ch_22epochs_bs8.pth locally and on Drive
Saved loss curves plot to teacher_loss_curves_full.png and /content/drive/MyDrive/Computational Imaging Project/checkpoints/teacher_loss_curves_full.png


<Figure size 640x480 with 0 Axes>

In [ ]:
%run train_student_kd.py

--- [train_student_kd] Setting up system path...
--- [train_student_kd] Added /content/CSC2529-Fall-Project/models/external/MambaIR to sys.path

--- [train_student_kd] Configuration ---
Train Dir: /content/dataset/UDC-SIT/training
Val Dir:   /content/dataset/UDC-SIT/validation
Patch Size: 256, Batch Size: 64
Epochs: 20, Learning Rate: 0.0002
Device: cuda
Teacher Weights: teacher_4ch_22epochs_bs8.pth
Local latest student ckpt: student_4ch_latest.pth
Drive latest student ckpt: /content/drive/MyDrive/Computational Imaging Project/checkpoints/student_latest.pth
-----------------------------------

--- [train_student_kd] Loading datasets...
--- [UDCDataset] Loaded 1864 files from /content/dataset/UDC-SIT/training
--- [UDCDataset] Loaded 238 files from /content/dataset/UDC-SIT/validation
--- [train_student_kd] Device: cuda
--- [train_student_kd] Loading teacher model from teacher_4ch_22epochs_bs8.pth...
--- [Teacher] Initializing with 4 in-channels and 4 out-channels.
--- [FrequencyDomainBlo

/content/CSC2529-Fall-Project/train_student_kd.py:134: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
--- [train_student_kd] Starting Knowledge Distillation training...


Epoch 1/20:   0%|          | 0/30 [00:00<?, ?it/s]/content/CSC2529-Fall-Project/train_student_kd.py:155: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/CSC2529-Fall-Project/models/frequency_block.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=False):
/content/CSC2529-Fall-Project/train_student_kd.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/20: 100%|██████████| 30/30 [10:40<00:00, 21.35s/it]

--- [train_student_kd] Epoch 1 Train Loss: 2.3520


...

--- [train_student_kd] Epoch 19 Val Charbonnier Loss: 0.0380
Saved latest student checkpoint to student_4ch_latest.pth and /content/drive/MyDrive/Computational Imaging Project/checkpoints/student_latest.pth


Epoch 20/20: 100%|██████████| 30/30 [10:31<00:00, 21.04s/it]

--- [train_student_kd] Epoch 20 Train Loss: 1.3328


--- [train_student_kd] Epoch 20 Val Charbonnier Loss: 0.0374
Saved latest student checkpoint to student_4ch_latest.pth and /content/drive/MyDrive/Computational Imaging Project/checkpoints/student_latest.pth
New best val loss. Saved best student checkpoint to /content/drive/MyDrive/Computational Imaging Project/checkpoints/student_best.pth
--- [train_student_kd] Final student saved as student_distilled_4ch_full_data.pth locally and on Drive
Saved loss curves plot to student_loss_curves_full.png and /content/drive/MyDrive/Computational Imaging Project/checkpoints/student_loss_curves_full.png


<Figure size 640x480 with 0 Axes>

In [ ]:
!git pull origin model2

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 7.71 KiB | 3.86 MiB/s, done.
From https://github.com/navdeepsingh1609/CSC2529-Fall-Project
 * branch            model2     -> FETCH_HEAD
   620b30d..385ed43  model2     -> origin/model2
Updating 620b30d..385ed43
Fast-forward
 eval_srgb_udc.py | 534 ++++++++++++++++++++++++-------------------------------
 testing_udc.py   | 503 +++++++++++++++++++++++++++------------------------
 viz_srgb_udc.py  | 392 +++++++++++++++++++---------------------
 3 files changed, 685 insertions(+), 744 deletions(-)


In [ ]:
# Quick training (small subset, for debugging)
!python train_teacher.py \
  --data-root /content/dataset/UDC-SIT \
  --preset quick \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_quick"

# Full training (long run on full train split)
!python train_teacher.py \
  --data-root /content/dataset/UDC-SIT \
  --preset full \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_full"


In [ ]:
# Quick KD (using the quick teacher)
!python train_student_kd.py \
  --data-root /content/dataset/UDC-SIT \
  --preset quick \
  --teacher-weights checkpoints_quick/teacher_4ch_quick_final.pth \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_quick"

# Full KD
!python train_student_kd.py \
  --data-root /content/dataset/UDC-SIT \
  --preset full \
  --teacher-weights checkpoints_full/teacher_4ch_full_final.pth \
  --save-loss-history \
  --drive-checkpoint-dir "/content/drive/MyDrive/Computational Imaging Project/checkpoints_full"


In [ ]:
# 1) Raw-domain testing + .npy predictions
!python testing_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split validation \
  --max-images 300 \
  --results-root results_final \
  --drive-results-root "/content/drive/MyDrive/Computational Imaging Project/results_final"
  # --use-tiling

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

--- [testing_udc] Configuration ---
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          validation
Patch size:     256
Patch batch:    8
Max images:     300
Teacher ckpt:   teacher_4ch_22epochs_bs8.pth
Student ckpt:   student_distilled_4ch_full_data.pth
Results root:   results_final
Drive results:  /content/drive/MyDrive/Computational Imaging Project/results_final
Use tiling:     False
Save NPY:       True
Device:         cuda
-----------------------------------

Setting up LPIPS (VGG) on device: cuda
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since

In [ ]:
# 2) sRGB metrics (input/teacher/student vs GT)
!python eval_srgb_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split validation \
  --teacher-pred-dir results_final/teacher_full_validation/npy \
  --student-pred-dir results_final/student_full_validation/npy \
  --results-root results_srgb_metrics \
  --drive-results-root "/content/drive/MyDrive/Computational Imaging Project/results_srgb_metrics_final" \
  --wb-mode none \
  --gamma 1.0
  # --wb-mode channel_gains \
  # --gamma 2.2


=== [eval_srgb_udc] Config ===
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          validation
Teacher preds:  results_final/teacher_full_validation/npy
Student preds:  results_final/student_full_validation/npy
Max images:     None
Results root:   results_srgb_metrics
Drive results:  /content/drive/MyDrive/Computational Imaging Project/results_srgb_metrics_final
WB mode:        none
Gamma:          1.0
[eval_srgb_udc] Saved CSV to results_srgb_metrics/validation_metrics_srgb.csv
[eval_srgb_udc] Saved summary to results_srgb_metrics/validation_metrics_srgb_summary.txt
Summary (mean metrics):
  Input   -> PSNR: 20.7550 dB, SSIM: 0.7349
  Teacher -> PSNR: 22.9563 dB, SSIM: 0.7879
  Student -> PSNR: 23.0665 dB, SSIM: 0.7490
[eval_srgb_udc] Copied results to Drive: /content/drive/MyDrive/Computational Imaging Project/results_srgb_metrics_final


In [ ]:
# 3) Visualization panels (Input / Teacher / Student / GT)
!python viz_srgb_udc.py \
  --data-root /content/dataset/UDC-SIT/UDC-SIT \
  --split validation \
  --teacher-pred-dir results_quick/teacher_full_validation/npy \
  --student-pred-dir results_quick/student_full_validation/npy \
  --results-root results_srgb_viz \
  --drive-results-root "/content/drive/MyDrive/Computational Imaging Project/results_srgb_viz" \
  --wb-mode none \
  --gamma 1.0
  # --wb-mode channel_gains \
  # --gamma 2.2

=== [viz_srgb_udc] Config ===
Data root:      /content/dataset/UDC-SIT/UDC-SIT
Split:          validation
Teacher preds:  results_quick/teacher_full_validation/npy
Student preds:  results_quick/student_full_validation/npy
Max images:     50
Viz root:       results_srgb_viz
Drive viz root: /content/drive/MyDrive/Computational Imaging Project/results_srgb_viz
WB mode:        none
Gamma:          1.0
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1004_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1027_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1034_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1081_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1127_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1132_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/114_panel.png
[viz_srgb_udc] Saved panel: results_srgb_viz/validation/1155_panel.png
[viz_srgb_udc] Saved panel: resu

In [ ]:
import os
import shutil
import random
from glob import glob

import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from skimage.metrics import peak_signal_noise_ratio


# ============================================================
# CONFIG – EDIT THESE PATHS FOR YOUR SANDBOX REPO
# ============================================================
# 4-channel .npy directories
INPUT_DIR_4CH   = "./input_npy"     # e.g. /content/input_4ch
GT_DIR_4CH      = "./gt_npy"        # e.g. /content/gt_4ch
TEACHER_DIR_4CH = "./teacher_npy"   # e.g. /content/teacher_pred_4ch
STUDENT_DIR_4CH = "./student_npy"   # e.g. /content/student_pred_4ch

# Path to official script + background
VISUALIZE_SIT_PY   = "./visualize_sit.py"
BACKGROUND_DNG     = "./background.dng"   # not used explicitly here, but visualize_sit.py should

# How many random samples to look at
N_SAMPLES = 5
RANDOM_SEED = 42

# Where to drop temporary / output stuff
TMP_ROOT           = "./_tmp_udc_vis"
OFFICIAL_SRGB_ROOT = "./srgb_official"
NAIVE_SRGB_ROOT    = "./srgb_naive"

# Which methods to run
RUN_METHOD_OFFICIAL      = True   # Method 1: visualize_sit.py
RUN_METHOD_NAIVE         = True   # Method 2: naive Bayer→RGB
RUN_METHOD_NAIVE_WB_GAMMA = True  # Method 3: naive + WB + gamma
# ============================================================


# ------------------ Utility helpers -------------------------

def get_common_filenames():
    """
    Return sorted list of basenames (e.g. '0001.npy') that exist
    in all four dirs: input, gt, teacher, student.
    """
    input_files   = {os.path.basename(p) for p in glob(os.path.join(INPUT_DIR_4CH, "*.npy"))}
    gt_files      = {os.path.basename(p) for p in glob(os.path.join(GT_DIR_4CH, "*.npy"))}
    teacher_files = {os.path.basename(p) for p in glob(os.path.join(TEACHER_DIR_4CH, "*.npy"))}
    student_files = {os.path.basename(p) for p in glob(os.path.join(STUDENT_DIR_4CH, "*.npy"))}

    common = sorted(input_files & gt_files & teacher_files & student_files)
    if not common:
        raise RuntimeError("No common .npy filenames found across input/gt/teacher/student dirs.")
    return common


def load_npy_4ch(dir_path, fname):
    """
    Load a 4-channel UDC-SIT .npy file.
    Returns float32 array in [0, 1023] shaped (H, W, 4).
    """
    path = os.path.join(dir_path, fname)
    arr = np.load(path)
    if arr.ndim != 3 or arr.shape[2] != 4:
        raise ValueError(f"{path} is not (H, W, 4); got shape {arr.shape}")
    return arr.astype(np.float32)


def naive_bayer4_to_rgb(arr4):
    """
    Quick-and-dirty 4ch → RGB mapping.
    Assumes:
      ch0, ch3: greens, ch1: red, ch2: blue
    Returns float32 RGB in [0,1], shape (H, W, 3).
    """
    arr_norm = arr4 / 1023.0  # [0,1]

    R = arr_norm[..., 1]
    G = 0.5 * (arr_norm[..., 0] + arr_norm[..., 3])
    B = arr_norm[..., 2]

    rgb = np.stack([R, G, B], axis=-1)
    rgb = np.clip(rgb, 0.0, 1.0)
    return rgb


def apply_wb_gamma(rgb, wb=(1.1, 1.0, 1.3), gamma=2.2):
    """
    Apply per-channel white-balance and gamma correction to [0,1] RGB.
    wb: (wr, wg, wb) multipliers.
    gamma: typical ~2.2 (we apply 1/gamma power).
    """
    rgb_wb = rgb * np.array(wb, dtype=np.float32)[None, None, :]
    rgb_wb = np.clip(rgb_wb, 0.0, 1.0)
    rgb_gamma = np.power(rgb_wb, 1.0 / gamma)
    return np.clip(rgb_gamma, 0.0, 1.0)


def show_panel(input_img, teacher_img, student_img, gt_img, title):
    """
    Simple matplotlib panel: Input | Teacher | Student | GT.
    Expects all images as (H, W, 3) in [0,1] or uint8.
    """
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    imgs = [input_img, teacher_img, student_img, gt_img]
    labels = ["Input", "Teacher", "Student", "GT"]

    for ax, im, lab in zip(axes, imgs, labels):
        if im.dtype != np.uint8:
            ax.imshow(np.clip(im, 0.0, 1.0))
        else:
            ax.imshow(im)
        ax.set_title(lab)
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def psnr_srgb_uint8(pred, gt):
    """PSNR between two uint8 sRGB images (H, W, 3)."""
    return peak_signal_noise_ratio(gt, pred, data_range=255)


def psnr_rgb_float01(pred, gt):
    """PSNR between two float32 [0,1] RGB images (H, W, 3)."""
    return peak_signal_noise_ratio(gt, pred, data_range=1.0)


# ------------------ Method 1: Official ISP ------------------

def run_visualize_sit_for_subset(role, src_dir, fnames):
    """
    Copy the subset of 4ch .npy into a temp folder and run visualize_sit.py
    on that folder.

    role: 'input' | 'gt' | 'teacher' | 'student'
    src_dir: directory containing .npy
    fnames: list of basenames (e.g. '0001.npy')
    """
    subset_npy_dir = os.path.join(TMP_ROOT, f"{role}_npy")
    out_srgb_dir   = os.path.join(OFFICIAL_SRGB_ROOT, f"{role}_srgb")

    os.makedirs(subset_npy_dir, exist_ok=True)
    os.makedirs(out_srgb_dir, exist_ok=True)

    # Copy just the selected files
    for fname in fnames:
        src = os.path.join(src_dir, fname)
        dst = os.path.join(subset_npy_dir, fname)
        shutil.copy(src, dst)

    # Call official script (assumes background.dng is findable by visualize_sit.py)
    cmd = f"python {VISUALIZE_SIT_PY} --data-directory \"{subset_npy_dir}\" --result-directory \"{out_srgb_dir}\""
    print(f"[Method 1] Running:\n  {cmd}")
    os.system(cmd)

    return out_srgb_dir


def method_official_isp(fnames):
    """
    Uses visualize_sit.py to produce sRGB PNGs from 4ch .npy
    and computes PSNR in the sRGB domain.
    """
    print("\n=== Method 1: Official ISP via visualize_sit.py ===")
    os.makedirs(TMP_ROOT, exist_ok=True)
    os.makedirs(OFFICIAL_SRGB_ROOT, exist_ok=True)

    # Run official script for each role
    input_srgb_dir   = run_visualize_sit_for_subset("input",   INPUT_DIR_4CH,   fnames)
    gt_srgb_dir      = run_visualize_sit_for_subset("gt",      GT_DIR_4CH,      fnames)
    teacher_srgb_dir = run_visualize_sit_for_subset("teacher", TEACHER_DIR_4CH, fnames)
    student_srgb_dir = run_visualize_sit_for_subset("student", STUDENT_DIR_4CH, fnames)

    teacher_psnrs = []
    student_psnrs = []

    for fname in fnames:
        base = os.path.splitext(fname)[0]
        png_name = base + ".png"

        inp_png  = os.path.join(input_srgb_dir,   png_name)
        gt_png   = os.path.join(gt_srgb_dir,      png_name)
        tch_png  = os.path.join(teacher_srgb_dir, png_name)
        stu_png  = os.path.join(student_srgb_dir, png_name)

        if not (os.path.exists(inp_png) and os.path.exists(gt_png) and
                os.path.exists(tch_png) and os.path.exists(stu_png)):
            print(f"[Method 1] Missing PNG(s) for {fname}, skipping.")
            continue

        inp_img  = imageio.imread(inp_png)   # uint8
        gt_img   = imageio.imread(gt_png)
        tch_img  = imageio.imread(tch_png)
        stu_img  = imageio.imread(stu_png)

        # PSNR
        psnr_t = psnr_srgb_uint8(tch_img, gt_img)
        psnr_s = psnr_srgb_uint8(stu_img, gt_img)
        teacher_psnrs.append(psnr_t)
        student_psnrs.append(psnr_s)

        print(f"[Method 1] {fname}: "
              f"Teacher PSNR={psnr_t:.2f} dB, Student PSNR={psnr_s:.2f} dB")

        # Show visual panel (scaled to [0,1] for matplotlib)
        show_panel(inp_img / 255.0, tch_img / 255.0, stu_img / 255.0, gt_img / 255.0,
                   title=f"Official ISP - {base}")

    if teacher_psnrs:
        print(f"\n[Method 1] Mean Teacher PSNR (sRGB): {np.mean(teacher_psnrs):.3f} dB")
        print(f"[Method 1] Mean Student PSNR (sRGB): {np.mean(student_psnrs):.3f} dB")
    else:
        print("[Method 1] No valid images for PSNR.")


# ------------- Method 2 & 3: Naive RGB ----------------------

def method_naive_rgb(fnames, use_wb_gamma=False):
    """
    Naive 4ch → RGB visualization (no rawpy), optionally with WB+gamma.
    Computes PSNR in this pseudo-sRGB space and shows panels.
    """
    label = "Method 2: Naive Bayer→RGB"
    if use_wb_gamma:
        label = "Method 3: Naive→RGB + WB + Gamma"
    print(f"\n=== {label} ===")

    teacher_psnrs = []
    student_psnrs = []

    for fname in fnames:
        base = os.path.splitext(fname)[0]

        arr_inp = load_npy_4ch(INPUT_DIR_4CH,   fname)
        arr_gt  = load_npy_4ch(GT_DIR_4CH,      fname)
        arr_tch = load_npy_4ch(TEACHER_DIR_4CH, fname)
        arr_stu = load_npy_4ch(STUDENT_DIR_4CH,fname)

        rgb_inp = naive_bayer4_to_rgb(arr_inp)
        rgb_gt  = naive_bayer4_to_rgb(arr_gt)
        rgb_tch = naive_bayer4_to_rgb(arr_tch)
        rgb_stu = naive_bayer4_to_rgb(arr_stu)

        if use_wb_gamma:
            rgb_inp = apply_wb_gamma(rgb_inp)
            rgb_gt  = apply_wb_gamma(rgb_gt)
            rgb_tch = apply_wb_gamma(rgb_tch)
            rgb_stu = apply_wb_gamma(rgb_stu)

        psnr_t = psnr_rgb_float01(rgb_tch, rgb_gt)
        psnr_s = psnr_rgb_float01(rgb_stu, rgb_gt)
        teacher_psnrs.append(psnr_t)
        student_psnrs.append(psnr_s)

        print(f"[{label}] {fname}: "
              f"Teacher PSNR={psnr_t:.2f} dB, Student PSNR={psnr_s:.2f} dB")

        show_panel(rgb_inp, rgb_tch, rgb_stu, rgb_gt,
                   title=f"{label} - {base}")

    if teacher_psnrs:
        print(f"\n[{label}] Mean Teacher PSNR: {np.mean(teacher_psnrs):.3f} dB")
        print(f"[{label}] Mean Student PSNR: {np.mean(student_psnrs):.3f} dB")
    else:
        print(f"[{label}] No valid images for PSNR.")


# ----------------------------- MAIN -------------------------

def main():
    print("Collecting common filenames...")
    common = get_common_filenames()
    print(f"Found {len(common)} common .npy files.")

    random.seed(RANDOM_SEED)
    n = min(N_SAMPLES, len(common))
    samples = sorted(random.sample(common, n))
    print(f"Using {n} sample(s):")
    for s in samples:
        print("  -", s)

    # Method 1: official ISP / sRGB via visualize_sit.py
    if RUN_METHOD_OFFICIAL:
        method_official_isp(samples)

    # Method 2: naive Bayer→RGB
    if RUN_METHOD_NAIVE:
        method_naive_rgb(samples, use_wb_gamma=False)

    # Method 3: naive Bayer→RGB + WB + gamma
    if RUN_METHOD_NAIVE_WB_GAMMA:
        method_naive_rgb(samples, use_wb_gamma=True)


if __name__ == "__main__":
    main()
